# Promoções recentes e comentadas do Pelando

Este notebook combina o feed de ofertas recentes (`/feed/v2/recents`) com o feed exibido em [pelando.com.br/comentadas](https://www.pelando.com.br/comentadas), servido por `/feed/v2/last-commented`. Os dois feeds são paginados por `pageInfo.endCursor` e as ofertas repetidas são consolidadas pelo ID.

A coluna `comentarios_por_hora` mede a média de comentários desde o cadastro até a execução da coleta. `score_engajamento` combina essa taxa (peso 66%) e a temperatura (peso 33%), ambos normalizados por min-max. `score_destaque` aplica ao score um fator de recência com meia-vida de 24 horas. A seção de destaques mostra as cinco promoções com maior `score_destaque`; a tabela geral continua ordenada pela data de cadastro mais recente.

A resposta da API não fornece um preço original explícito. `preco_referencia` é estimado a partir de `discountFixed` ou `discountPercentage`, quando disponível, e fica vazio quando não há dados suficientes. `sourceUrl` é preferido como link de compra; `redirectUrl` é usado como alternativa. Ao concluir a coleta, o notebook gera `pelando_ofertas.html` com todas as ofertas, incluindo a decomposição do score, foto e link para compra.

Executado localmente, o notebook pode publicar o HTML no GitHub Pages usando um token informado interativamente. No GitHub Actions, a publicação é feita pelo workflow agendado, sem solicitar ou armazenar um token pessoal.


In [23]:
import base64
import os
import time

from getpass import getpass

from html import escape
from pathlib import Path
from urllib.parse import quote, urlsplit

import pandas as pd
from curl_cffi import requests
from IPython.display import display

In [ ]:
API_BASE = "https://api-web.pelando.com.br"
FEEDS = {
    "recentes": {
        "endpoint": "/feed/v2/recents",
        "referer": "https://www.pelando.com.br/",
    },
    "mais_comentadas": {
        "endpoint": "/feed/v2/last-commented",
        "referer": "https://www.pelando.com.br/comentadas",
    },
}
PAGE_SIZE = 50
MAX_PAGES = 10  # até 500 ofertas por feed; aumente para buscar mais páginas
REQUEST_TIMEOUT = (5, 30)
PAUSE_BETWEEN_PAGES = 0.25

# Cabeçalhos necessários para reproduzir a consulta pública observada no site.
HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://www.pelando.com.br",
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    ),
}

In [ ]:
def buscar_feed(config, nome_feed, page_size=PAGE_SIZE, max_pages=MAX_PAGES):
    """Busca páginas sequenciais de um feed público do Pelando."""
    if page_size < 1 or max_pages < 1:
        raise ValueError("page_size e max_pages devem ser maiores que zero.")

    ofertas = []
    cursor = None
    cursores_usados = set()
    chegou_ao_fim = False
    endpoint = API_BASE + config["endpoint"]
    headers = {**HEADERS, "Referer": config["referer"]}

    with requests.Session(impersonate="chrome136") as session:
        session.headers.update(headers)
        for numero_pagina in range(max_pages):
            params = {"limit": page_size}
            if cursor is not None:
                params["after"] = cursor

            resposta = session.get(
                endpoint, params=params, timeout=REQUEST_TIMEOUT
            )
            resposta.raise_for_status()
            payload = resposta.json()
            dados = payload.get("data") if isinstance(payload, dict) else None
            if not isinstance(dados, dict):
                raise ValueError(f"Resposta inválida no feed {nome_feed!r}: sem 'data'.")

            pagina = dados.get("deals")
            if not isinstance(pagina, list):
                raise ValueError(f"Resposta inválida no feed {nome_feed!r}: sem 'data.deals'.")
            ofertas.extend(pagina)

            page_info = dados.get("pageInfo") or {}
            if not page_info.get("hasNextPage"):
                chegou_ao_fim = True
                break

            proximo_cursor = page_info.get("endCursor")
            if not proximo_cursor or proximo_cursor in cursores_usados:
                raise RuntimeError(f"Cursor ausente ou repetido no feed {nome_feed!r}.")
            cursores_usados.add(proximo_cursor)
            cursor = proximo_cursor
            if numero_pagina + 1 < max_pages:
                time.sleep(PAUSE_BETWEEN_PAGES)

    if not chegou_ao_fim:
        print(
            f"Limite de {max_pages} páginas atingido no feed {nome_feed!r}; "
            "aumente MAX_PAGES para buscar mais ofertas."
        )
    return ofertas


def estimar_preco_referencia(oferta):
    """Estima o preço anterior usando o desconto informado pela API."""
    preco = oferta.get("price")
    if preco is None:
        return None

    preco = float(preco)
    desconto_fixo = oferta.get("discountFixed")
    if desconto_fixo is not None and float(desconto_fixo) > 0:
        return round(preco + float(desconto_fixo), 2)

    desconto_percentual = oferta.get("discountPercentage")
    if desconto_percentual is not None:
        percentual = float(desconto_percentual)
        if 0 < percentual < 100:
            return round(preco / (1 - percentual / 100), 2)
    return None


def transformar_oferta(oferta):
    loja = oferta.get("store") or {}
    return {
        "id": oferta.get("id"),
        "nome": oferta.get("title"),
        "foto": oferta.get("imageUrl"),
        "valor": oferta.get("price"),
        "link_compra": oferta.get("sourceUrl") or oferta.get("redirectUrl"),
        "preco_referencia": estimar_preco_referencia(oferta),
        "data_cadastro": oferta.get("createdAt"),
        "loja": loja.get("name"),
        "tipo": oferta.get("kind"),
        "temperatura": oferta.get("temperature"),
        "numero_comentarios": oferta.get("commentCount"),
        "desconto_fixo": oferta.get("discountFixed"),
        "desconto_percentual": oferta.get("discountPercentage"),
    }


def buscar_e_consolidar_ofertas(page_size=PAGE_SIZE, max_pages=MAX_PAGES):
    """Une os feeds recente e comentado, consolidando ofertas repetidas pelo ID."""
    por_id = {}
    for nome_feed, config in FEEDS.items():
        ofertas_feed = buscar_feed(config, nome_feed, page_size, max_pages)
        for oferta in ofertas_feed:
            registro = transformar_oferta(oferta)
            oferta_id = registro["id"]
            if not oferta_id:
                raise ValueError(f"Oferta sem ID recebida no feed {nome_feed!r}.")

            if oferta_id not in por_id:
                por_id[oferta_id] = {**registro, "origens": [nome_feed]}
                continue

            existente = por_id[oferta_id]
            if nome_feed not in existente["origens"]:
                existente["origens"].append(nome_feed)
            for coluna in ("temperatura", "numero_comentarios"):
                valores = [existente[coluna], registro[coluna]]
                valores_validos = [valor for valor in valores if valor is not None]
                existente[coluna] = max(valores_validos) if valores_validos else None
    return list(por_id.values())

GITHUB_REPO_NAME = "radar-promocoes-pelando"
GITHUB_PAGES_BRANCH = "gh-pages"
GITHUB_API_BASE = "https://api.github.com"
GITHUB_API_VERSION = "2026-03-10"


def publicar_github_pages(arquivo_html):
    """Create/reuse the public repo, upload index.html, and configure Pages."""
    token = getpass(
        "GitHub token (classic PAT with repo scope; hidden input): "
    ).strip()
    if not token:
        raise ValueError("GitHub token is empty; publication was cancelled.")

    def api_request(method, endpoint, allowed_statuses, json_body=None, params=None):
        response = requests.request(
            method,
            f"{GITHUB_API_BASE}{endpoint}",
            headers={
                "Accept": "application/vnd.github+json",
                "Authorization": f"Bearer {token}",
                "X-GitHub-Api-Version": GITHUB_API_VERSION,
            },
            json=json_body,
            params=params,
            timeout=30,
        )
        if response.status_code not in allowed_statuses:
            try:
                message = response.json().get("message", response.text[:500])
            except ValueError:
                message = response.text[:500]
            raise RuntimeError(
                f"GitHub API {method} {endpoint} returned HTTP "
                f"{response.status_code}: {message}. Check token permissions (repo) "
                "and GitHub Pages availability."
            )
        if not response.content:
            return response.status_code, None
        try:
            return response.status_code, response.json()
        except ValueError as exc:
            raise RuntimeError(
                f"GitHub API {method} {endpoint} did not return valid JSON."
            ) from exc

    try:
        _, profile = api_request("GET", "/user", (200,))
        owner = profile.get("login") if isinstance(profile, dict) else None
        if not owner:
            raise RuntimeError("Could not identify the authenticated GitHub account.")

        repo_endpoint = f"/repos/{owner}/{GITHUB_REPO_NAME}"
        repo_status, repo = api_request("GET", repo_endpoint, (200, 404))
        if repo_status == 404:
            _, repo = api_request(
                "POST",
                "/user/repos",
                (201,),
                json_body={
                    "name": GITHUB_REPO_NAME,
                    "description": "Public dashboard of recent Pelando deals",
                    "private": False,
                    "auto_init": True,
                },
            )
            print(f"Created public repository: https://github.com/{owner}/{GITHUB_REPO_NAME}")
        elif repo.get("private"):
            raise RuntimeError(
                f"https://github.com/{owner}/{GITHUB_REPO_NAME} exists but is private. "
                "Change its visibility before publishing on GitHub Pages."
            )

        default_branch = repo.get("default_branch") or "main"
        default_ref_path = (
            f"{repo_endpoint}/git/ref/heads/{quote(default_branch, safe='')}"
        )
        branch_endpoint = f"{repo_endpoint}/git/ref/heads/{GITHUB_PAGES_BRANCH}"
        branch_status, _ = api_request("GET", branch_endpoint, (200, 404))
        if branch_status == 404:
            _, source_ref = api_request("GET", default_ref_path, (200,))
            api_request(
                "POST",
                f"{repo_endpoint}/git/refs",
                (201,),
                json_body={
                    "ref": f"refs/heads/{GITHUB_PAGES_BRANCH}",
                    "sha": source_ref["object"]["sha"],
                },
            )

        html_path = Path(arquivo_html)
        if not html_path.is_file():
            raise FileNotFoundError(f"HTML file not found: {html_path}")
        if html_path.stat().st_size >= 100 * 1024 * 1024:
            raise ValueError("File exceeds the GitHub Contents API limit of 100 MB.")

        contents_endpoint = f"{repo_endpoint}/contents/index.html"
        content_status, current_file = api_request(
            "GET", contents_endpoint, (200, 404), params={"ref": GITHUB_PAGES_BRANCH}
        )
        upload_body = {
            "message": "Update Pelando promotions dashboard",
            "content": base64.b64encode(html_path.read_bytes()).decode("ascii"),
            "branch": GITHUB_PAGES_BRANCH,
        }
        if content_status == 200:
            upload_body["sha"] = current_file["sha"]
        api_request("PUT", contents_endpoint, (200, 201), json_body=upload_body)

        pages_endpoint = f"{repo_endpoint}/pages"
        pages_status, _ = api_request("GET", pages_endpoint, (200, 404))
        pages_config = {
            "build_type": "legacy",
            "source": {"branch": GITHUB_PAGES_BRANCH, "path": "/"},
        }
        if pages_status == 404:
            api_request("POST", pages_endpoint, (201,), json_body=pages_config)
        else:
            api_request("PUT", pages_endpoint, (200, 204), json_body=pages_config)

        _, pages = api_request("GET", pages_endpoint, (200,))
        site_url = pages.get("html_url") or f"https://{owner}.github.io/{GITHUB_REPO_NAME}/"
        print(f"Uploaded index.html to branch {GITHUB_PAGES_BRANCH}.")
        print(f"GitHub Pages URL: {site_url}")
        print("The first deployment can take a few minutes to become available.")
        return site_url
    finally:
        token = ""


In [27]:
dados_brutos = buscar_e_consolidar_ofertas()
df = pd.DataFrame(dados_brutos)

if df.empty:
    print("A API não retornou ofertas.")
else:
    df["data_cadastro"] = pd.to_datetime(
        df["data_cadastro"], utc=True, errors="coerce"
    ).dt.tz_convert("America/Sao_Paulo")
    datas_invalidas = int(df["data_cadastro"].isna().sum())
    if datas_invalidas:
        print(f"Aviso: {datas_invalidas} oferta(s) sem data de cadastro válida.")

    for coluna in ("numero_comentarios", "temperatura"):
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce").fillna(0)

    def normalizar_min_max(serie):
        menor, maior = serie.min(), serie.max()
        if maior == menor:
            return pd.Series(0.0, index=serie.index)
        return (serie - menor) / (maior - menor)

    agora_execucao = pd.Timestamp.now(tz="UTC")
    data_cadastro_utc = df["data_cadastro"].dt.tz_convert("UTC")
    df["idade_horas"] = (
        (agora_execucao - data_cadastro_utc).dt.total_seconds() / 3600
    ).clip(lower=0)
    idade_minima_horas = 1 / 60  # evita divisão por zero em ofertas recém-publicadas
    df["comentarios_por_hora"] = (
        df["numero_comentarios"] / df["idade_horas"].clip(lower=idade_minima_horas)
    )
    df["fator_recencia"] = 2 ** (-df["idade_horas"] / 24)  # meia-vida de 24 horas

    comentarios_normalizados = normalizar_min_max(df["comentarios_por_hora"])
    temperatura_normalizada = normalizar_min_max(df["temperatura"])
    df["comentarios_por_hora"] = df["comentarios_por_hora"].round(4)
    df["comentarios_normalizados"] = comentarios_normalizados.round(4)
    df["temperatura_normalizada"] = temperatura_normalizada.round(4)
    df["contribuicao_comentarios_score"] = (
        comentarios_normalizados * (66 / 99) * 100
    ).round(2)
    df["contribuicao_temperatura_score"] = (
        temperatura_normalizada * (33 / 99) * 100
    ).round(2)
    df["score_engajamento"] = (
        df["contribuicao_comentarios_score"]
        + df["contribuicao_temperatura_score"]
    ).round(2)
    df["score_destaque"] = (
        df["score_engajamento"] * df["fator_recencia"]
    ).round(2)

    # Data é a chave primária; score só desempata ofertas com a mesma data.
    df = df.sort_values(
        ["data_cadastro", "score_engajamento"],
        ascending=[False, False],
        na_position="last",
        kind="mergesort",
    ).reset_index(drop=True)

    def url_segura(valor):
        if not isinstance(valor, str):
            return ""
        partes = urlsplit(valor)
        return valor if partes.scheme in {"http", "https"} else ""

    def texto_html(valor):
        return "—" if pd.isna(valor) else escape(str(valor), quote=True)

    def moeda_html(valor):
        if pd.isna(valor):
            return "—"
        formatado = f"{float(valor):,.2f}"
        return "R$ " + formatado.replace(",", "X").replace(".", ",").replace("X", ".")

    linhas_html = []
    for _, oferta in df.iterrows():
        foto_url = url_segura(oferta["foto"])
        nome = texto_html(oferta["nome"])
        loja = texto_html(oferta["loja"])
        foto_html = (
            f'<img class="thumb" src="{escape(foto_url, quote=True)}" '
            f'alt="Foto de {nome}" loading="lazy">'
            if foto_url
            else '<span class="thumb-empty">&mdash;</span>'
        )
        compra_url = url_segura(oferta["link_compra"])
        link_html = (
            f'<a class="buy-button" href="{escape(compra_url, quote=True)}" '
            'target="_blank" rel="noopener noreferrer">Ver oferta &rarr;</a>'
            if compra_url
            else '<span class="no-link">Indisponível</span>'
        )
        data = oferta["data_cadastro"]
        data_texto = data.strftime("%d/%m/%Y %H:%M") if pd.notna(data) else "&mdash;"
        criado_em = int(data.timestamp() * 1000) if pd.notna(data) else 0
        score = float(oferta["score_engajamento"])
        score_destaque = float(oferta["score_destaque"])
        comentarios_por_hora = float(oferta["comentarios_por_hora"])
        preco = float(oferta["valor"]) if pd.notna(oferta["valor"]) else -1
        comentarios = int(oferta["numero_comentarios"])
        temperatura = int(oferta["temperatura"])
        contribuicao_comentarios = float(oferta["contribuicao_comentarios_score"])
        contribuicao_temperatura = float(oferta["contribuicao_temperatura_score"])
        score_aria = (
            f"Score {score:.1f} de 100; comentários contribuem "
            f"{contribuicao_comentarios:.1f}; temperatura contribui {contribuicao_temperatura:.1f}"
        )
        linhas_html.append(f"""
        <tr class="offer-row" data-name="{nome.lower()}" data-store="{loja.lower()}"
          data-created="{criado_em}" data-score="{score}" data-highlight="{score_destaque}"
          data-comments="{comentarios}" data-comments-rate="{comentarios_por_hora}"
          data-temperature="{temperatura}" data-price="{preco}">
          <td class="photo-cell">{foto_html}</td>
          <td class="product-cell"><strong>{nome}</strong><span>{loja}</span></td>
          <td class="price-cell"><strong>{moeda_html(oferta['valor'])}</strong>
            <span>Ref. {moeda_html(oferta['preco_referencia'])}</span></td>
          <td class="date-cell" data-label="Cadastro">{texto_html(data_texto)}<span>GMT-3</span></td>

          <td class="score-cell" data-label="Engajamento">
            <div class="score-number" aria-label="{escape(score_aria, quote=True)}">{score:.1f}<small>/100</small></div>
            <div class="score-track" role="meter" aria-label="Score de engajamento" aria-valuemin="0" aria-valuemax="100" aria-valuenow="{score:.1f}">
              <span style="width:{min(100, max(0, score)):.2f}%"></span></div>
            <span class="score-breakdown" title="Comentários por hora normalizados (peso 66): {contribuicao_comentarios:.2f} pontos &middot; Temperatura (peso 33): {contribuicao_temperatura:.2f} pontos">
              Coment. {contribuicao_comentarios:.1f} &middot; Temp. {contribuicao_temperatura:.1f}
            </span>
          </td>
          <td class="number-cell" data-label="Comentários">&#128172; {comentarios}<span>{comentarios_por_hora:.2f}/h</span></td>
          <td class="number-cell" data-label="Temperatura">&#128293; {temperatura}&#176;</td>          <td class="action-cell">{link_html}</td>
        </tr>""")

    destaques = (
        df[df["tipo"] == "promotion"]
        .nlargest(10, "score_destaque")
    )
    destaques_html = []
    for _, oferta in destaques.iterrows():
        foto_url = url_segura(oferta["foto"])
        foto_destaque = (
            f'<img class="highlight-thumb" src="{escape(foto_url, quote=True)}" '
            f'alt="" loading="lazy">'
            if foto_url
            else '<span class="highlight-thumb-empty">—</span>'
        )
        compra_url = url_segura(oferta["link_compra"])
        link_destaque = (
            f'<a href="{escape(compra_url, quote=True)}" target="_blank" '
            'rel="noopener noreferrer">Ver oferta ↗</a>'
            if compra_url
            else '<span>Link indisponível</span>'
        )
        destaques_html.append(
            f'<article class="highlight-item">{foto_destaque}'
            f'<div class="highlight-info"><strong>{texto_html(oferta["nome"])}</strong>'
            f'<span>{float(oferta["comentarios_por_hora"]):.2f} comentários/h · '
            f'{int(oferta["numero_comentarios"])} comentários · {float(oferta["idade_horas"]):.1f} h · fator de recência {float(oferta["fator_recencia"]):.2f}</span></div>'
            f'<div class="highlight-score"><strong>{float(oferta["score_destaque"]):.1f}</strong>'
            f'<span>score</span></div>{link_destaque}</article>'
        )
    destaque_vazio = '<p class="empty">Nenhuma promoção disponível para destacar.</p>'
    execucao_local = agora_execucao.tz_convert("America/Sao_Paulo").strftime("%d/%m/%Y %H:%M")

    documento_html = f"""<!doctype html>
<html lang="pt-BR">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <meta name="color-scheme" content="light">
  <title>Ofertas do Pelando</title>
  <style>
    :root {{ color-scheme:light; --ink:#202938; --muted:#758095; --line:#e5e9f0; --brand:#e9513b; --accent:#5266dc; }}
    * {{ box-sizing:border-box; }}
    body {{ margin:0; color:var(--ink); background:#f5f7fa; font:14px/1.45 Inter,ui-sans-serif,system-ui,-apple-system,"Segoe UI",sans-serif; }}
    .page {{ max-width:1420px; margin:0 auto; padding:clamp(16px,3vw,36px); }}
    header {{ display:flex; align-items:flex-end; justify-content:space-between; gap:16px; margin-bottom:22px; }}
    .eyebrow {{ color:var(--brand); font-size:11px; font-weight:800; letter-spacing:.12em; text-transform:uppercase; }}
    h1 {{ margin:4px 0; font-size:clamp(25px,3vw,34px); line-height:1.2; letter-spacing:-.035em; }}
    .subtitle {{ margin:4px 0 0; color:var(--muted); }}
    .count {{ flex:none; padding:7px 12px; border:1px solid var(--line); border-radius:999px; background:#fff; color:var(--muted); font-weight:700; }}
    .toolbar {{ display:flex; gap:10px; margin-bottom:12px; }}
    input,select {{ min-height:42px; border:1px solid var(--line); border-radius:9px; background:white; color:var(--ink); padding:0 12px; font:inherit; outline:none; }}
    input {{ flex:1; min-width:120px; }} select {{ min-width:210px; cursor:pointer; }}
    input:focus,select:focus {{ border-color:var(--accent); box-shadow:0 0 0 3px #5266dc20; }}
    .table-shell {{ overflow-x:auto; border:1px solid var(--line); border-radius:12px; background:#fff; box-shadow:0 3px 14px #25334b0a; }}
    table {{ width:100%; min-width:900px; border-collapse:separate; border-spacing:0; }}
    th {{ position:sticky; top:0; z-index:1; padding:11px 12px; background:#f8f9fb; color:#667187; font-size:11px; font-weight:800; letter-spacing:.045em; text-align:left; text-transform:uppercase; border-bottom:1px solid var(--line); }}
    td {{ padding:9px 12px; border-bottom:1px solid #edf0f4; vertical-align:middle; }}
    tbody tr:last-child td {{ border-bottom:0; }} tbody tr:hover {{ background:#fafbfe; }}
    .photo-cell {{ width:66px; }} .thumb {{ display:block; width:48px; height:48px; border:1px solid #edf0f4; border-radius:8px; background:#fff; object-fit:contain; }}
    .thumb-empty {{ display:grid; place-items:center; width:48px; height:48px; border:1px dashed var(--line); border-radius:8px; color:#a0a8b5; }}
    .product-cell {{ min-width:190px; max-width:360px; }} .product-cell strong {{ display:-webkit-box; overflow:hidden; -webkit-box-orient:vertical; -webkit-line-clamp:2; font-size:13px; line-height:1.4; }}
    .product-cell span,.price-cell span,.date-cell span {{ display:block; margin-top:3px; color:var(--muted); font-size:11px; }}
    .price-cell {{ white-space:nowrap; }} .price-cell strong {{ color:#263d78; font-size:14px; }}
    .date-cell,.number-cell {{ white-space:nowrap; font-size:12px; }} .date-cell span {{ font-size:10px; }} .number-cell {{ font-weight:700; }}
    .score-cell {{ width:178px; min-width:160px; }} .score-number {{ color:var(--accent); font-size:15px; font-weight:800; }} .score-number small {{ margin-left:2px; color:#929bad; font-size:10px; font-weight:600; }}
    .score-track {{ width:100%; height:6px; margin:4px 0; overflow:hidden; border-radius:99px; background:#e9ecf4; }} .score-track span {{ display:block; height:100%; border-radius:inherit; background:linear-gradient(90deg,#5479ed,#835de7); }}
    .score-breakdown {{ color:var(--muted); font-size:10px; white-space:nowrap; }}
    .number-cell span {{ display:block; margin-top:2px; color:var(--muted); font-size:10px; font-weight:500; }}
    .highlights {{ margin:18px 0; padding:16px; border:1px solid #e2e8fb; border-radius:12px; background:linear-gradient(120deg,#fff,#f8f9ff); }}
    .highlights h2 {{ margin:0; font-size:17px; }} .highlights-note {{ margin:3px 0 12px; color:var(--muted); font-size:11px; }}
    .highlight-list {{ display:grid; gap:7px; }}
    .highlight-item {{ display:grid; grid-template-columns:42px minmax(0,1fr) 58px auto; align-items:center; gap:10px; padding:7px; border:1px solid var(--line); border-radius:9px; background:#fff; }}
    .highlight-thumb,.highlight-thumb-empty {{ width:40px; height:40px; border:1px solid #edf0f4; border-radius:7px; object-fit:contain; }}
    .highlight-thumb-empty {{ display:grid; place-items:center; color:var(--muted); }}
    .highlight-info {{ display:grid; min-width:0; gap:2px; }} .highlight-info strong {{ overflow:hidden; text-overflow:ellipsis; white-space:nowrap; font-size:12px; }} .highlight-info span {{ color:var(--muted); font-size:10px; }}
    .highlight-score {{ display:grid; justify-items:center; color:var(--accent); }} .highlight-score strong {{ font-size:17px; }} .highlight-score span {{ color:var(--muted); font-size:9px; }}
    .highlight-item>a {{ color:#c54430; font-size:11px; font-weight:800; text-decoration:none; white-space:nowrap; }} .highlight-item>span {{ color:var(--muted); font-size:10px; }}
    .buy-button {{ display:inline-flex; align-items:center; justify-content:center; min-height:32px; padding:0 10px; border-radius:8px; background:#fff2ef; color:#c54430; font-size:12px; font-weight:800; text-decoration:none; white-space:nowrap; }} .buy-button:hover {{ background:#ffe3dd; }} .no-link {{ color:#a0a8b5; font-size:11px; }}
    [hidden] {{ display:none !important; }} .empty {{ padding:28px; color:var(--muted); text-align:center; }}
    @media(max-width:700px) {{ .page {{ padding:16px 12px; }} header {{ align-items:flex-start; }} .subtitle {{ max-width:38ch; }} .count {{ font-size:12px; }} .toolbar {{ flex-direction:column; }} select {{ width:100%; }} .table-shell {{ overflow:visible; border:0; background:transparent; box-shadow:none; }} table,tbody {{ display:block; min-width:0; }} thead {{ display:none; }} tbody {{ display:grid; gap:8px; }} .offer-row {{ display:grid; grid-template-columns:58px minmax(0,1fr) auto; gap:0 8px; padding:9px; border:1px solid var(--line); border-radius:10px; background:#fff; box-shadow:0 2px 8px #25334b08; }} td {{ padding:3px 0; border:0; }} .photo-cell {{ grid-column:1; grid-row:1 / span 2; align-self:start; }} .thumb,.thumb-empty {{ width:52px; height:52px; }} .product-cell {{ grid-column:2 / 4; grid-row:1; min-width:0; max-width:none; }} .price-cell {{ grid-column:2; grid-row:2; padding-top:5px; }} .date-cell {{ grid-column:3; grid-row:2; padding-top:5px; text-align:right; }} .score-cell {{ grid-column:1 / 3; grid-row:3; width:auto; min-width:0; padding-top:8px; }} .number-cell {{ grid-row:4; padding-top:8px; }} .number-cell:nth-of-type(6) {{ grid-column:1 / 2; }} .number-cell:nth-of-type(7) {{ grid-column:2 / 3; }} .action-cell {{ grid-column:3; grid-row:3 / span 2; align-self:center; justify-self:end; }} .score-breakdown {{ font-size:9px; }} .highlight-item {{ grid-template-columns:36px minmax(0,1fr) 48px; gap:7px; }} .highlight-thumb,.highlight-thumb-empty {{ width:34px; height:34px; }} .highlight-item>a,.highlight-item>span {{ grid-column:2 / 4; }} .highlight-item>a {{ padding-top:3px; }} }}
    @media(prefers-reduced-motion:reduce) {{ *,*::before,*::after {{ scroll-behavior:auto !important; transition:none !important; }} }}
  </style>
</head>
<body>
  <main class="page">
    <header><div><div class="eyebrow">Pelando &middot; ofertas</div>
      <h1>Ofertas recentes e comentadas</h1>
      <p class="subtitle">Compare preço e engajamento. A lista começa pelas ofertas mais recentes.</p></div>
      <div class="count"><span id="visible-count">{len(df)}</span> de {len(df)} ofertas</div>
    </header>
    <section class="highlights" aria-labelledby="highlights-title">
      <h2 id="highlights-title">Destaques: conversa e recência</h2>
      <p class="highlights-note">Top 10 promoções por score de destaque = score de engajamento × fator de recência (meia-vida de 24 h). Atualizado em {execucao_local} (GMT-3).</p>
      <div class="highlight-list">{''.join(destaques_html) or destaque_vazio}</div>
    </section>
    <section class="toolbar" aria-label="Filtros de ofertas">
      <input id="search" type="search" placeholder="Buscar produto ou loja..." aria-label="Buscar produto ou loja">
      <select id="sort" aria-label="Ordenar ofertas">
        <option value="created">Mais recentes</option>
        <option value="score">Maior score</option>
        <option value="highlight">Maior score de destaque</option>
        <option value="comments-rate">Mais comentários por hora</option>
        <option value="comments">Mais comentadas</option>
        <option value="temperature">Maior temperatura</option>
        <option value="price-asc">Menor preço</option>
        <option value="price-desc">Maior preço</option>
      </select>
    </section>
    <div class="table-shell"><table aria-label="Ofertas do Pelando">
      <thead><tr><th>Foto</th><th>Produto</th><th>Preço</th><th>Cadastro (GMT-3)</th><th>Score de engajamento</th><th>Comentários</th><th>Temperatura</th><th></th></tr></thead>
      <tbody id="offers">{''.join(linhas_html)}</tbody>
    </table><div id="empty" class="empty" hidden>Nenhuma oferta corresponde à busca.</div></div>
  </main>
  <script>
    const body = document.getElementById('offers');
    const search = document.getElementById('search');
    const sort = document.getElementById('sort');
    const count = document.getElementById('visible-count');
    const empty = document.getElementById('empty');
    const rows = [...body.querySelectorAll('.offer-row')];
    const sortKeys = {{ created:'created', score:'score', highlight:'highlight', 'comments-rate':'commentsRate', comments:'comments', temperature:'temperature' }};
    function updateOffers() {{
      const query = search.value.trim().toLocaleLowerCase('pt-BR');
      const key = sort.value;
      const ascending = key === 'price-asc';
      const metric = key.startsWith('price-') ? 'price' : sortKeys[key];
      rows.sort((a,b) => {{
        const left = Number(a.dataset[metric]);
        const right = Number(b.dataset[metric]);
        const result = ascending ? left-right : right-left;
        return result || Number(b.dataset.created)-Number(a.dataset.created);
      }});
      let visible = 0;
      rows.forEach(row => {{
        const matches = (row.dataset.name+' '+row.dataset.store).includes(query);
        row.hidden = !matches;
        if (matches) visible += 1;
        body.appendChild(row);
      }});
      count.textContent = visible;
      empty.hidden = visible !== 0;
    }}
    search.addEventListener('input', updateOffers);
    sort.addEventListener('change', updateOffers);
    updateOffers();
  </script>
</body>
</html>"""
    arquivo_html = Path.cwd() / "pelando_ofertas.html"
    arquivo_html.write_text(documento_html, encoding="utf-8")
    if os.getenv("GITHUB_ACTIONS") == "true":
        url_publicada = None
        print("Publicacao sera feita pelo workflow do GitHub Pages.")
    else:
        url_publicada = publicar_github_pages(arquivo_html)

    print(f"Ofertas unicas coletadas: {len(df)}")
    print(f"Arquivo HTML gerado: {arquivo_html.resolve()}")
    if url_publicada:
        print(f"URL publicada: {url_publicada}")
    display(df.head(100))

Limite de 10 páginas atingido no feed 'recentes'; aumente MAX_PAGES para buscar mais ofertas.
Limite de 10 páginas atingido no feed 'mais_comentadas'; aumente MAX_PAGES para buscar mais ofertas.
Created public repository: https://github.com/gabrielsouza-oss/radar-promocoes-pelando
Uploaded index.html to branch gh-pages.
GitHub Pages URL: https://gabrielsouza-oss.github.io/radar-promocoes-pelando/
The first deployment can take a few minutes to become available.
Ofertas unicas coletadas: 758
Arquivo HTML gerado: C:\Users\gabri\OneDrive\Desktop\Scrapping Promoções\pelando_ofertas.html
URL publicada: https://gabrielsouza-oss.github.io/radar-promocoes-pelando/


,id,nome,foto,valor,link_compra,preco_referencia,data_cadastro,loja,tipo,temperatura,...,origens,idade_horas,comentarios_por_hora,fator_recencia,comentarios_normalizados,temperatura_normalizada,contribuicao_comentarios_score,contribuicao_temperatura_score,score_engajamento,score_destaque
0,e9ab5f24-ee34-4ce6-9ba5-c260bf158cd3,Jogo Panelas Tramontina Allegra Inox Fundo Tri...,https://media.pelando.com.br/x4pNyBJa5WU5LVigt...,375.00,https://www.amazon.com.br/dp/B09TCD67Q3,None,2026-11-07 13:11:52.313000-03:00,Amazon,promotion,17,...,"[recentes, mais_comentadas]",0.122488,8.1640,0.996469,0.2540,0.0239,16.93,0.80,17.73,17.67
1,19790b7b-1e15-48cd-88cd-cd55ab6e4b9a,Relógio Masculino Invicta Speedway Quartzo 42mm,https://media.pelando.com.br/6XI5uxI0TIGNvyARN...,239.00,https://a.co/d/07rQniUi,None,2026-11-07 12:58:29.142000-03:00,Amazon,promotion,107,...,"[recentes, mais_comentadas]",0.345592,2.8936,0.990069,0.0900,0.0324,6.00,1.08,7.08,7.01
2,4ccd18cc-9b13-48a8-b4e0-f1c5e64b5f4a,Kit Lavagem Automotiva Completo Vonixx,https://media.pelando.com.br/Z-YWQoYAs3eTVD1o4...,92.00,https://a.co/d/0g6GVnSx,None,2026-11-07 12:57:30.502000-03:00,Amazon,promotion,46,...,[recentes],0.361880,0.0000,0.989603,0.0000,0.0266,0.00,0.89,0.89,0.88
3,4f615b00-42d3-48b0-94f8-eaa5cf0aafe6,"[R$14 +POR-] Salgadinho de Batata Lay's, Origi...",https://media.pelando.com.br/YYJPUBSVyl-bprQE0...,15.89,https://a.co/d/01ZwAenf,None,2026-11-07 12:51:00.760000-03:00,Amazon,promotion,48,...,"[recentes, mais_comentadas]",0.470142,14.8891,0.986514,0.4632,0.0268,30.88,0.89,31.77,31.34
4,8d4064e0-e8df-4010-b76a-92a1852b39c9,(Regional) Atum Ralado Burguesa Híbrido 120g,https://media.pelando.com.br/WGPMXIqcHz6VOsk_U...,7.00,https://www.efacil.com.br/loja/produto/atum-ra...,None,2026-11-07 12:50:56.971000-03:00,eFácil,promotion,32,...,"[recentes, mais_comentadas]",0.471195,4.2445,0.986484,0.1320,0.0253,8.80,0.84,9.64,9.51
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,825b9110-36ee-43fb-a188-9102fc6133b5,Kit Desodorante Twist Herbíssimo Clinical Men 45G,https://media.pelando.com.br/c_D_ZJLVAhP0WOowT...,22.90,https://shopee.com.br/product/436104412/182969...,None,2026-11-07 08:22:56.818000-03:00,Shopee,promotion,34,...,[recentes],4.937904,0.0000,0.867090,0.0000,0.0255,0.00,0.85,0.85,0.74
96,6d7cd5eb-d6c5-4d34-9da9-4f7b6a411d6e,WHISKAS Ração Carne Gatos Adultos 900 g,https://media.pelando.com.br/okZ35z5cs6so6tIZb...,13.59,https://a.co/d/0cz8bqtg,None,2026-11-07 08:18:04.091000-03:00,Amazon,promotion,193,...,"[recentes, mais_comentadas]",5.019217,0.7969,0.865056,0.0248,0.0404,1.65,1.35,3.00,2.60
97,d3a82beb-984d-4db3-ad60-e4e1092c5d37,Cupom Shopee - R$10 OFF acima de R$40,https://media.pelando.com.br/jieKWR8-kR0Rh9n5Y...,NaN,https://shopee.com.br/user/voucher-wallet?mmp_...,None,2026-11-07 08:02:46.419000-03:00,Shopee,coupon,131,...,"[recentes, mais_comentadas]",5.274126,1.7064,0.858711,0.0531,0.0346,3.54,1.15,4.69,4.03
98,27038d30-9717-4c1b-bd3e-f8f3d8a5d45d,Camisa Juvenil Spfc 2024 Comemorativa Torcedor,https://media.pelando.com.br/2h3UKBKNk2xJXD-5g...,69.00,https://www.amazon.com.br/dp/B0DLCBXWTK?,None,2026-11-07 08:00:12.953000-03:00,Amazon,promotion,92,...,[recentes],5.316755,0.0000,0.857655,0.0000,0.0310,0.00,1.03,1.03,0.88


A variável `df` contém as ofertas consolidadas dos dois feeds. `comentarios_por_hora` é a média desde o cadastro até o instante da coleta, com idade mínima de um minuto para evitar divisão por zero; a API fornece a contagem total, não uma série temporal de comentários. O score de engajamento combina taxa normalizada de comentários (66%) e temperatura normalizada (33%); o score de destaque multiplica esse valor por um fator de recência cuja meia-vida é 24 horas. O HTML gerado contém uma seção com as cinco promoções mais destacadas, seguida da tabela completa, que pode ser buscada e ordenada por data, score, destaque, comentários/hora, temperatura ou preço. Para coletar mais páginas de cada feed, aumente `MAX_PAGES`.